In [1]:
import torch
from torch import nn
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
from os.path import join
from sklearn.metrics import accuracy_score
from torch.optim import SGD
import numpy as np
from pytorch_lightning.loggers import TensorBoardLogger

In [2]:
NUM_EPOCHS = 150

In [3]:
import pytorch_lightning as pl

# Autoencoder

In [4]:
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torchvision import transforms

transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize((0.1307,), (0.3081,)),
                                torch.flatten]) #flatten serve per convertire le immagini in vettori di 784 unità 

mnist_train = MNIST(root='mnist',train=True, download=True, transform=transform)
mnist_test = MNIST(root='mnist',train=False, download=True, transform=transform)


mnist_train_loader = DataLoader(mnist_train, batch_size=1024, num_workers=2, shuffle=True)
mnist_test_loader = DataLoader(mnist_test, batch_size=1024, num_workers=2)


In [5]:
#print shape of the data
print(mnist_train.data.shape)

torch.Size([60000, 28, 28])


In [6]:
from torchvision.utils import make_grid

class Autoencoder(pl.LightningModule):
    def __init__(self):
        super(Autoencoder, self).__init__() 

        self.training_step_outputs = []   # save outputs in each batch to compute metric overall epoch
        self.training_step_targets = []   # save targets in each batch to compute metric overall epoch
        self.val_step_outputs = []        # save outputs in each batch to compute metric overall epoch
        self.val_step_targets = [] 
        
        #l'encoder è un MLP che mappa l'input in un codice di 128 unità
        #dato che l'ultimo livello dell'MLP non è l'ultimo della rete,
        #inseriamo una attivazione alla fine dell'MLP
        self.encoder = nn.Sequential(nn.Linear(784, 256),
                                     nn.ReLU(),
                                     nn.Linear(256, 128),
                                     nn.ReLU())
        
        #il decoder è un MLP che mappa il codice in un output di dimensione uguale all'input
        self.decoder = nn.Sequential(nn.Linear(128, 256),
                                     nn.ReLU(),
                                     nn.Linear(256, 784))
        
        #la loss che utilizzeremo per il training
        self.criterion = nn.MSELoss()
        
    def forward(self, x):
        code = self.encoder(x)
        reconstructed = self.decoder(code)
        #restituiamo sia il codice che l'output ricostruito
        return code, reconstructed
    
    # questo metodo definisce l'optimizer
    def configure_optimizers(self):
        optimizer = SGD(self.parameters(), lr=0.01, momentum=0.99) 
        return optimizer
    
    # questo metodo definisce come effettuare ogni singolo step di training
    def training_step(self, train_batch, batch_idx):
        x, _ = train_batch
        _, reconstructed = self.forward(x)
        loss = self.criterion(x, reconstructed)
        
        self.training_step_outputs.extend(reconstructed)
        self.training_step_targets.extend(x)

        self.log('train/loss', loss)
        return loss
    
    def on_train_epoch_end(self):
        self.training_step_outputs = self.training_step_outputs[0].view(-1,1,28,28)[:50,...]
        self.training_step_targets = self.training_step_targets[0].view(-1,1,28,28)[:50,...]
        self.logger.experiment.add_image('input_images', make_grid(self.training_step_targets, nrow=10, normalize=True),self.global_step)
        self.logger.experiment.add_image('generated_images', make_grid(self.training_step_outputs, nrow=10, normalize=True),self.global_step)

        self.training_step_outputs = []
        self.training_step_targets = []
    
    # questo metodo definisce come effettuare ogni singolo step di validation
    def validation_step(self, val_batch, batch_idx):
        x, _ = val_batch
        _, reconstructed = self.forward(x)
        loss = self.criterion(x, reconstructed)

        self.val_step_outputs.extend(reconstructed)
        self.val_step_targets.extend(x)
        
        self.log('val/loss', loss)
        if batch_idx==0:
            return {'inputs':x, 'outputs':reconstructed}
        
    def on_validation_epoch_end(self):
        self.val_step_outputs = self.val_step_outputs[0].view(-1,1,28,28)[:50,...]
        self.val_step_targets = self.val_step_targets[0].view(-1,1,28,28)[:50,...]
        
        self.logger.experiment.add_image('input_images', make_grid(self.val_step_targets, nrow=10, normalize=True),self.global_step)
        self.logger.experiment.add_image('generated_images', make_grid(self.val_step_outputs, nrow=10, normalize=True),self.global_step)

        self.val_step_outputs = []
        self.val_step_targets = []
        

logger = TensorBoardLogger("tb_logs", name="autoencoder")
    
mnist_autoencoder = Autoencoder()
trainer = pl.Trainer(max_epochs=NUM_EPOCHS, accelerator="gpu", devices=1, logger=logger) 
trainer.fit(mnist_autoencoder, mnist_train_loader, mnist_test_loader)


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

  | Name      | Type       | Params
-----------------------------------------
0 | encoder   | Sequential | 233 K 
1 | decoder   | Sequential | 234 K 
2 | criterion | MSELoss    | 0     
-----------------------------------------
468 K     Trainable params
0         Non-trainable params
468 K     Total params
1.873     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:432: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x1122deb90>
Traceback (most recent call last):
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py", line 1443, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/multiprocessing/connection.py", line 931, in wait
    ready = selector.select(timeout)
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/selectors.py", 

Epoch 1: 100%|██████████| 59/59 [00:02<00:00, 28.80it/s, v_num=5]

Exception in thread Thread-4:
Traceback (most recent call last):
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/tensorboard/summary/writer/event_file_writer.py", line 244, in run
    self._run()
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/tensorboard/summary/writer/event_file_writer.py", line 275, in _run
    self._record_writer.write(data)
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/tensorboard/summary/writer/record_writer.py", line 40, in write
    self._writer.write(header + header_crc + data + footer_crc)
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.10/site-packages/tensorboard/compat/tensorflow_stub/io/gfile.py", line 773, in write
    self.fs.append(self.filename, file_content, self.binary_mode)
  File "/Users/ale/miniforge3/envs/pytorch-env/lib/python3.

FileNotFoundError: [Errno 2] No such file or directory: b'tb_logs/autoencoder/version_5/events.out.tfevents.1716473129.MacBookessandra.station.60664.0'